# Numerical CPP: structure / embedding / fused arms

`CPP.run_num` runs the **same** algorithm as `CPP.run`, but the per-residue values come from a pre-sliced numerical tensor (`dict_num_parts`) instead of an amino-acid→scale lookup. The key idea:

- **Your PLM embeddings ARE the `dict_num`.** A `{entry: (L, D)}` tensor (ProtT5, ESM, …) feeds straight in — there is no conversion step.
- `EmbeddingPreprocessor.build_pseudo_scales` / `cluster_pseudo_scales` only **name** the `D` dimensions (producing `df_scales` / `df_cat`); they are *not* a per-residue value source.
- The same entry point serves three arms — **embedding**, **structure-only**, and **fused** — differing only in which `dict_num` you pass.

In [ ]:
import numpy as np
import aaanalysis as aa

aa.options["verbose"] = False
df_seq = aa.load_dataset(name="DOM_GSEC", n=10)
labels = df_seq["label"].to_list()

# Stand-in for a real PLM: a per-residue embedding tensor {entry: (L, D)}.
D = 8
rng = np.random.default_rng(0)
dict_emb = {e: rng.random((len(s), D)) for e, s in zip(df_seq["entry"], df_seq["sequence"])}
list(dict_emb.items())[0][0], list(dict_emb.values())[0].shape

## Embedding arm

`build_pseudo_scales` / `cluster_pseudo_scales` turn the embedding corpus into `df_scales` / `df_cat` that **name** the `D` dimensions. `NumericalFeature.get_parts` slices both the sequence parts and the tensor with shared boundaries; `CPP.run_num` then consumes the tensor directly.

In [ ]:
ep = aa.EmbeddingPreprocessor()
df_scales_emb = ep.build_pseudo_scales(df_seq=df_seq, dict_num=dict_emb)   # (20, D) — names the dims
df_cat_emb = ep.cluster_pseudo_scales(df_scales_emb=df_scales_emb)         # (D, 5) categories

nf = aa.NumericalFeature()
df_parts, dict_num_parts = nf.get_parts(df_seq=df_seq, dict_num=dict_emb)

cpp = aa.CPP(df_parts=df_parts, df_scales=df_scales_emb, df_cat=df_cat_emb, verbose=False)
df_feat_emb = cpp.run_num(dict_num_parts=dict_num_parts, labels=labels, n_filter=10, n_jobs=1)
df_feat_emb.head()

## Fused arm

To combine sources (e.g. embeddings **and** structure features), concatenate the per-residue tensors along the `D` axis with `aa.combine_dict_nums` *first*, then run the identical pipeline. Here we add a 3-dim structure-like tensor to the 8-dim embedding for `D = 11`.

In [ ]:
dict_struct = {e: rng.random((len(s), 3)) for e, s in zip(df_seq["entry"], df_seq["sequence"])}
dict_fused = aa.combine_dict_nums([dict_emb, dict_struct])   # D = 8 + 3 = 11

df_scales_f = ep.build_pseudo_scales(df_seq=df_seq, dict_num=dict_fused)
df_cat_f = ep.cluster_pseudo_scales(df_scales_emb=df_scales_f)
df_parts_f, dnp_f = nf.get_parts(df_seq=df_seq, dict_num=dict_fused)

cpp_f = aa.CPP(df_parts=df_parts_f, df_scales=df_scales_f, df_cat=df_cat_f, verbose=False)
df_feat_fused = cpp_f.run_num(dict_num_parts=dnp_f, labels=labels, n_filter=10, n_jobs=1)
print("fused D =", df_scales_f.shape[1])
df_feat_fused.head()

**Structure-only** works the same way — pass a `dict_num` from `StructurePreprocessor` (DSSP / PDB / PAE encoders) instead of the embedding tensor. In every arm the only thing that changes is the `dict_num`; the `get_parts` → `run_num` pipeline and the `df_feat` output schema are identical to sequence-mode `CPP.run`.